# 03 - Streaming
Traitement temps reel avec Spark Structured Streaming

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os, time, threading, shutil

spark = SparkSession.builder \
    .appName("03_Streaming") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

# Charger la dimension locations (statique)
locations = spark.read.parquet("hdfs://namenode:9000/projet/silver/locations_clean/")

# Recuperer le schema depuis Silver
trips_schema = spark.read.parquet("hdfs://namenode:9000/projet/silver/trips_clean/").schema

print("Spark connecte")
print(f"Colonnes: {trips_schema.names}")

In [ ]:
# Simulateur de flux : decoupe le CSV en micro-batches
stream_input = "/home/jovyan/data/stream_input"

# Nettoyer le dossier si il existe deja
if os.path.exists(stream_input):
    shutil.rmtree(stream_input)
os.makedirs(stream_input)

def simulateur():
    import pandas as pd
    reader = pd.read_csv("/home/jovyan/data/fact_trips_sample.csv", chunksize=500)
    for i, chunk in enumerate(reader):
        chunk.to_csv(f"{stream_input}/batch_{i:04d}.csv", index=False)
        time.sleep(3)
        if i >= 50:
            break
    print("Simulateur termine")

thread = threading.Thread(target=simulateur)
thread.start()
print("Simulateur lance (50 batches x 500 lignes, 1 batch toutes les 3s)")

In [ ]:
# Consumer Streaming
streaming_df = spark.readStream \
    .schema(trips_schema) \
    .option("header", True) \
    .option("maxFilesPerTrigger", 1) \
    .csv(stream_input)

# Enrichir avec les zones et agreger en temps reel
realtime_stats = (streaming_df
    .join(broadcast(locations),
          streaming_df.pickup_location_id == locations.location_id,
          "left")
    .groupBy("borough")
    .agg(
        count("*").alias("nb_trajets_live"),
        round(sum("total_amount"), 2).alias("revenu_live"),
        round(avg("trip_distance"), 2).alias("distance_moy_live")
    )
)

query = realtime_stats.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("live_stats") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Streaming demarre")

In [ ]:
# Voir les resultats (executez plusieurs fois, les chiffres augmentent)
spark.sql("SELECT * FROM live_stats ORDER BY nb_trajets_live DESC").show()

In [ ]:
# Re-executez pour voir l'evolution
spark.sql("SELECT * FROM live_stats ORDER BY nb_trajets_live DESC").show()

In [ ]:
# Re-executez une 3eme fois
spark.sql("SELECT * FROM live_stats ORDER BY nb_trajets_live DESC").show()

In [ ]:
# Arreter le streaming
query.stop()
print("Streaming arrete")